In [1]:
!git clone https://huggingface.co/enalis/scold
!cd scold

Cloning into 'scold'...
remote: Enumerating objects: 117, done.
remote: Total 117 (delta 0), reused 0 (delta 0), pack-reused 117 (from 1)
Receiving objects: 100% (117/117), 46.26 KiB | 6.61 MiB/s, done.
Resolving deltas: 100% (55/55), done.


In [2]:
import os
import sys
import gc
import logging
import re
import warnings
from pathlib import Path
from typing import List, Tuple, Optional, Dict, Any

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset
from PIL import Image
from torchvision import transforms
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import pandas as pd

# =============================================================================
# 0. Add SCOLD repo to path (adjust path to your cloned repository)
# =============================================================================
SCOLD_REPO_PATH = "./scold"
if os.path.exists(SCOLD_REPO_PATH):
    sys.path.insert(0, os.path.abspath(SCOLD_REPO_PATH))
else:
    raise FileNotFoundError(
        f"SCOLD repository not found at {SCOLD_REPO_PATH}. "
        "Please clone it: git clone https://huggingface.co/enalis/scold"
    )

try:
    from model import LVL
except ImportError as e:
    raise ImportError(
        "Failed to import LVL from model.py. "
        "Make sure you are running inside the cloned SCOLD directory."
    ) from e

# =============================================================================
# 1. Configuration (Kaggle / local friendly)
# =============================================================================
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Paths - Update this to your actual PlantVillage dataset location
DATA_PATH = "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color"  # <-- CHANGE THIS
SCOLD_CHECKPOINT = "/kaggle/working/scold/scold.pth"

# Model constants
EMBEDDING_DIM = 512  # SCOLD image embedding dimension
BATCH_SIZE = 32
NUM_WORKERS = 2

# FKP hyperparameters
P_DEFAULT = 64
ALPHA_RIDGE_DEFAULT = 1.0
BETA_RIDGE_DEFAULT = 1.0
PCLS_VAL_SPLIT = 0.2

# =============================================================================
# 2. Define Target Crops and Their Classes (from PlantVillage)
# =============================================================================
# Complete class list from PlantVillage for reference:
# Apple_* (0-3), Blueberry_healthy (5), Cherry_* (6-7), Corn_* (8-11),
# Grape_* (12-15), Orange_HLB (16), Peach_* (17-18), Pepper_* (19-20),
# Potato_* (21-23), Raspberry_healthy (24), Soybean_healthy (25),
# Squash_PM (26), Strawberry_* (27-28), Tomato_* (29-38)
# Source: AgML documentation and HuggingFace dataset card

CROP_CLASSES = {
    "Strawberry": {
        "indices": [27, 28],
        "names": ["Strawberry___Leaf_scorch", "Strawberry___healthy"]
    },
    "Tomato": {
        "indices": list(range(29, 39)),  # 29..38 inclusive
        "names": [
            "Tomato___Bacterial_spot", "Tomato___Early_blight", "Tomato___Late_blight",
            "Tomato___Leaf_Mold", "Tomato___Septoria_leaf_spot",
            "Tomato___Spider_mites Two-spotted_spider_mite",
            "Tomato___Target_Spot", "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
            "Tomato___Tomato_mosaic_virus", "Tomato___healthy"
        ]
    }
}

# =============================================================================
# 3. Dataset loading (unchanged)
# =============================================================================
def load_scold_model(checkpoint_path: str = SCOLD_CHECKPOINT, device: torch.device = DEVICE):
    """Load SCOLD (LVL) model from checkpoint."""
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(
            f"SCOLD checkpoint not found at {checkpoint_path}. "
            "Make sure scold.pth is present (Git LFS may need to be installed)."
        )
    model = LVL()
    state_dict = torch.load(checkpoint_path, map_location="cpu")
    model.load_state_dict(state_dict)
    model = model.to(device).eval()
    
    from transformers import RobertaTokenizer
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
    
    image_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])
    return model, tokenizer, image_transform

class PlantVillageDataset(Dataset):
    """PlantVillage dataset returning (image_tensor, label_index)."""
    def __init__(self, root_dir: str, transform: transforms.Compose):
        from torchvision.datasets import ImageFolder
        self.image_folder = ImageFolder(root_dir)
        self.transform = transform
        self.classes = self.image_folder.classes

    def __len__(self):
        return len(self.image_folder)

    def __getitem__(self, idx):
        img_path, label = self.image_folder.samples[idx]
        img = Image.open(img_path).convert("RGB")
        img_tensor = self.transform(img)
        return img_tensor, label

def extract_scold_embeddings(dataloader: DataLoader, model: nn.Module, device: torch.device) -> Tuple[np.ndarray, np.ndarray]:
    """Extract image embeddings using model.get_images_features()."""
    embeddings_list = []
    labels_list = []
    model.eval()
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Extracting SCOLD embeddings"):
            images = images.to(device)
            image_emb = model.get_images_features(images)
            embeddings_list.append(image_emb.cpu().numpy())
            labels_list.extend(labels.tolist())
    embeddings = np.concatenate(embeddings_list, axis=0)
    labels = np.array(labels_list)
    return embeddings, labels

# =============================================================================
# 4. FKP Core Functions
# =============================================================================
def compute_pcls(E: np.ndarray, H: np.ndarray, alpha: float = ALPHA_RIDGE_DEFAULT, val_split: float = PCLS_VAL_SPLIT) -> Tuple[float, Dict]:
    E_train, E_val, H_train, H_val = train_test_split(E, H, test_size=val_split, random_state=SEED)
    mean_E = E_train.mean(0, keepdims=True)
    mean_H = H_train.mean(0, keepdims=True)
    Ec_train = E_train - mean_E
    Hc_train = H_train - mean_H
    W = np.linalg.solve(Ec_train.T @ Ec_train + alpha * np.eye(Ec_train.shape[1]), Ec_train.T @ Hc_train)
    Ec_val = E_val - mean_E
    H_pred = Ec_val @ W + mean_H
    ss_res = np.sum((H_val - H_pred) ** 2)
    ss_tot = np.sum((H_val - mean_H) ** 2)
    pcls = 1 - ss_res / ss_tot if ss_tot > 0 else 1.0
    return pcls, {"mean_E": mean_E, "mean_H": mean_H, "W_ridge": W}

def rrp_fit(E: np.ndarray, H: np.ndarray, p: int, alpha: float = ALPHA_RIDGE_DEFAULT, beta: float = BETA_RIDGE_DEFAULT) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, Dict]:
    mean = E.mean(axis=0, keepdims=True).astype(np.float32)
    Ec = E.astype(np.float32) - mean
    Hc = H.astype(np.float32) - H.mean(axis=0, keepdims=True)
    W_ridge = np.linalg.solve(Ec.T @ Ec + alpha * np.eye(Ec.shape[1]), Ec.T @ Hc).astype(np.float32)
    Uw, sw, _ = np.linalg.svd(W_ridge, full_matrices=False)
    U = Uw[:, :p].astype(np.float32)
    Z = Ec @ U
    ridge = Ridge(alpha=beta, fit_intercept=True)
    ridge.fit(Z, H.astype(np.float32))
    W = ridge.coef_.T.astype(np.float32)
    b = ridge.intercept_.astype(np.float32)
    residual = np.linalg.norm(Hc - Ec @ W_ridge, 'fro') ** 2 / Ec.shape[0]
    tail_sq = np.sum(sw[p:] ** 2) if len(sw) > p else 0.0
    info = {
        "residual": residual,
        "tail_sq": tail_sq,
        "sv": sw,
        "sigma_p": sw[p - 1] if len(sw) >= p else 0.0,
        "sigma_p1": sw[p] if len(sw) > p else 0.0,
        "W_ridge_norm": np.linalg.norm(W_ridge, 'fro')
    }
    return U, mean, W, b, info

def evaluate_edge(E_test: np.ndarray, mean: np.ndarray, U: np.ndarray, W: np.ndarray, b: np.ndarray, Y_test: np.ndarray) -> float:
    Z = (E_test.astype(np.float32) - mean) @ U
    logits = Z @ W + b
    preds = np.argmax(logits, axis=1)
    return accuracy_score(Y_test, preds)

# =============================================================================
# 5. Per-Crop Training and Evaluation Function
# =============================================================================
def train_crop_head(crop_name: str, crop_class_info: Dict, full_dataset: Dataset, model: nn.Module, device: torch.device):
    """Extract subset for given crop, train FKP head, and report metrics."""
    print(f"\n{'='*60}")
    print(f"Processing crop: {crop_name}")
    print(f"Classes: {crop_class_info['names']}")
    print(f"Class indices: {crop_class_info['indices']}")

    # Collect indices belonging to this crop
    valid_idxs = []
    for idx, (_, label) in enumerate(full_dataset.image_folder.samples):
        if label in crop_class_info['indices']:
            valid_idxs.append(idx)
    print(f"Found {len(valid_idxs)} images for {crop_name}")

    if len(valid_idxs) < 100:
        print(f"⚠️ Warning: Only {len(valid_idxs)} images found. Results may be unreliable.")
        return None

    # Shuffle and split
    np.random.seed(SEED)
    np.random.shuffle(valid_idxs)
    calib_size = min(500, len(valid_idxs) // 2)
    test_size = min(500, len(valid_idxs) - calib_size)
    calib_idx = valid_idxs[:calib_size]
    test_idx = valid_idxs[calib_size:calib_size + test_size]

    calib_ds = Subset(full_dataset, calib_idx)
    test_ds = Subset(full_dataset, test_idx)

    calib_loader = DataLoader(calib_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    print("Extracting embeddings...")
    E_calib, Y_calib = extract_scold_embeddings(calib_loader, model, device)
    E_test, Y_test = extract_scold_embeddings(test_loader, model, device)

    # Remap labels to 0..num_classes-1 for this crop
    unique_labels = np.unique(np.concatenate([Y_calib, Y_test]))
    label_map = {old: new for new, old in enumerate(unique_labels)}
    Y_calib_mapped = np.array([label_map[y] for y in Y_calib])
    Y_test_mapped = np.array([label_map[y] for y in Y_test])

    num_classes = len(unique_labels)
    H_calib = np.eye(num_classes)[Y_calib_mapped]

    # Evaluate baseline: linear probe (teacher performance)
    from sklearn.linear_model import Ridge
    ridge_probe = Ridge(alpha=ALPHA_RIDGE_DEFAULT, fit_intercept=True)
    ridge_probe.fit(E_calib, H_calib)
    logits_test = E_test @ ridge_probe.coef_.T + ridge_probe.intercept_
    linear_probe_acc = accuracy_score(Y_test_mapped, np.argmax(logits_test, axis=1))

    # FKP training
    print(f"Training FKP head with p={P_DEFAULT}...")
    U, mean, W, b, info = rrp_fit(E_calib, H_calib, P_DEFAULT)
    acc_fkp = evaluate_edge(E_test, mean, U, W, b, Y_test_mapped)

    # Compute PCLS on calibration set
    pcls, _ = compute_pcls(E_calib, H_calib)

    # Print results
    print(f"\n📊 {crop_name.upper()} RESULTS")
    print(f"  Linear Probe (Teacher baseline): {linear_probe_acc*100:.2f}%")
    print(f"  FKP Accuracy (p={P_DEFAULT}): {acc_fkp*100:.2f}%")
    print(f"  PCLS (linearity score): {pcls:.4f}")
    print(f"  Model size: {U.size + W.size + b.size} parameters")

    # Save crop-specific heads for deployment
    np.save(f"{crop_name.lower()}_U.npy", U)
    np.save(f"{crop_name.lower()}_mean.npy", mean)
    np.save(f"{crop_name.lower()}_W.npy", W)
    np.save(f"{crop_name.lower()}_b.npy", b)
    print(f"💾 Saved {crop_name.lower()}_*.npy files for edge deployment.")

    return {
        "crop": crop_name,
        "samples": len(valid_idxs),
        "classes": num_classes,
        "linear_probe_acc": linear_probe_acc,
        "fkp_acc": acc_fkp,
        "pcls": pcls,
        "residual": info["residual"],
        "tail_sq": info["tail_sq"],
        "p": P_DEFAULT,
        "U_shape": U.shape,
        "W_shape": W.shape
    }

# =============================================================================
# 6. Main execution
# =============================================================================
def main():
    print("🌱 Seiwa Agriculture FKP Proof of Concept")
    print(f"Using device: {DEVICE}")

    # Validate dataset path
    if not os.path.exists(DATA_PATH):
        print(f"❌ Dataset path not found: {DATA_PATH}")
        print("Please update DATA_PATH to your PlantVillage location.")
        print("Example structure: /path/to/plantvillage/color/Apple___Apple_scab/")
        return

    print("Loading SCOLD model...")
    try:
        model, _, transform = load_scold_model()
    except Exception as e:
        print(f"❌ Failed to load SCOLD model: {e}")
        print("Make sure you have cloned the SCOLD repository and scold.pth is present.")
        return

    print("Loading full PlantVillage dataset...")
    full_dataset = PlantVillageDataset(DATA_PATH, transform)
    print(f"Total images: {len(full_dataset)}")
    print(f"Total classes: {len(full_dataset.classes)}")
    print("\nClasses found in dataset (first 10):")
    for i, cls in enumerate(full_dataset.classes[:10]):
        print(f"  {i}: {cls}")

    # Train crop-specific heads
    results = []
    for crop_name, crop_info in CROP_CLASSES.items():
        res = train_crop_head(crop_name, crop_info, full_dataset, model, DEVICE)
        if res:
            results.append(res)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Summary table
    print("\n" + "="*60)
    print("📈 FKP PERFORMANCE SUMMARY FOR SEIWA CROPS")
    print("="*60)
    summary_df = pd.DataFrame(results)
    print(summary_df.to_string(index=False))
    
    # Save results to Excel for presentation
    summary_df.to_excel("seiwa_poc_results.xlsx", index=False)
    print("\n💾 Results saved to seiwa_poc_results.xlsx")
    
    # Deployment memory estimate
    for crop in results:
        params = crop["U_shape"][0] * crop["U_shape"][1] + crop["W_shape"][0] * crop["W_shape"][1] + crop["W_shape"][1]
        print(f"\n📦 {crop['crop']} edge model: {params:,} params → {params*4/1024:.2f} KB (float32)")

if __name__ == "__main__":
    main()

Using device: cuda
🌱 Seiwa Agriculture FKP Proof of Concept
Using device: cuda
Loading SCOLD model...


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Extracting SCOLD embeddings:   0%|          | 0/16 [00:00<?, ?it/s]

Training FKP head with p=64...

📊 STRAWBERRY RESULTS
  Linear Probe (Teacher baseline): 100.00%
  FKP Accuracy (p=64): 100.00%
  PCLS (linearity score): 0.9612
  Model size: 1030 parameters
💾 Saved strawberry_*.npy files for edge deployment.

Processing crop: Tomato
Classes: ['Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus', 'Tomato___healthy']
Class indices: [29, 30, 31, 32, 33, 34, 35, 36, 37, 38]
Found 16033 images for Tomato
Extracting embeddings...


Extracting SCOLD embeddings:   0%|          | 0/16 [00:00<?, ?it/s]

Extracting SCOLD embeddings:   0%|          | 0/16 [00:00<?, ?it/s]

Training FKP head with p=64...

📊 TOMATO RESULTS
  Linear Probe (Teacher baseline): 97.20%
  FKP Accuracy (p=64): 97.20%
  PCLS (linearity score): 0.7800
  Model size: 4698 parameters
💾 Saved tomato_*.npy files for edge deployment.

📈 FKP PERFORMANCE SUMMARY FOR SEIWA CROPS
      crop  samples  classes  linear_probe_acc  fkp_acc     pcls  residual  tail_sq  p  U_shape W_shape
Strawberry     2583        2             1.000    1.000 0.961224  0.004251      0.0 64 (512, 2)  (2, 2)
    Tomato    16033        9             0.972    0.972 0.779969  0.130045      0.0 64 (512, 9)  (9, 9)

💾 Results saved to seiwa_poc_results.xlsx

📦 Strawberry edge model: 1,030 params → 4.02 KB (float32)

📦 Tomato edge model: 4,698 params → 18.35 KB (float32)
